<a href="https://colab.research.google.com/github/RRADJon/TEMPO/blob/main/AOP_NNfilter_Boltz2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
# @title Boltz-2 + NN 150s
ligand = "CC(=O)CC(C1=CC=CC=C1)C2=C(C3=CC=CC=C3OC2=O)O"

!pip install -q biopython httpx rdkit

import json, io, os, torch, base64, time, requests
import numpy as np
import pandas as pd
import torch.nn as nn
from pathlib import Path
from Bio.PDB.MMCIFParser import MMCIFParser
from scipy.spatial.distance import cdist
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from google.colab import userdata
from rdkit import Chem
from rdkit.Chem import Descriptors

# --- CONFIGURATION ---
NVIDIA_API_KEY = userdata.get('NVIDIA_API_KEY')
PUBLIC_URL = "https://health.api.nvidia.com/v1/biology/mit/boltz2/predict"
MODEL_B64 = """..."""
HEADERS = {"Authorization": f"Bearer {NVIDIA_API_KEY}", "Content-Type": "application/json"}

# --- 1. NEURAL NETWORK ARCHITECTURE ---
class BinderNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32), nn.BatchNorm1d(32), nn.ReLU(),
            nn.Dropout(0.4), nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 1), nn.Sigmoid()
        )
    def forward(self, x): return self.net(x)

# --- 2. PHYSICS & FEATURE EXTRACTION ---
parser = MMCIFParser(QUIET=True)

def extract_features_from_result(result_json):
    try:
        aff_root = result_json.get("affinity_predictions", result_json.get("affinities", {}))
        lig_aff = aff_root.get("L1", {})
        pic50 = lig_aff.get("affinity_pic50", [0.0])[0]
        prob = lig_aff.get("affinity_probability_binary", [0.0])[0]
        conf = np.mean(result_json.get("confidence_scores", [0.0]))

        cif_string = result_json["structures"][0]["structure"]
        structure = parser.get_structure("complex", io.StringIO(cif_string))
        p_coords = [a.get_coord() for a in structure.get_atoms() if len(a.get_parent().get_resname()) <= 3]
        l_coords = [a.get_coord() for a in structure.get_atoms() if len(a.get_parent().get_resname()) > 3 or a.get_parent().get_resname() == "LIG1"]

        if not p_coords or not l_coords: return None
        dists = cdist(np.array(p_coords), np.array(l_coords))

        bins = [np.sum((dists >= b[0]) & (dists < b[1])) for b in [(0, 2.5), (2.5, 3.5), (3.5, 4.5), (4.5, 6.0)]]
        clashes = np.sum(dists < 1.9)
        num_contacts = np.sum(dists < 4.5)

        return {
            "boltz_pic50": pic50, "boltz_prob": prob, "confidence": conf,
            "bin_0_2.5": bins[0], "bin_2.5_3.5": bins[1], "bin_3.5_4.5": bins[2], "bin_4.5_6.0": bins[3],
            "num_contacts": num_contacts, "clash_count": clashes
        }
    except Exception as e:
        print(f"Feature Extraction Error: {e}")
        return None

# --- 3. UTILITIES ---
def download_pdb(pdb_id):
    filename = f"{pdb_id}.pdb"
    if not os.path.exists(filename):
        url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
        r = requests.get(url)
        with open(filename, "wb") as f: f.write(r.content)
    return filename

def load_pdb_base64(path):
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

def predict_structure(protein_id, sequence, ligand_smiles, template_b64=None):
    request_body = {
        "polymers": [{
            "id": "A", "sequence": sequence, "molecule_type": "protein",
            "templates": [{"pdb": template_b64, "chain_id": "A"}] if template_b64 else [],
            "msa": {"uniref90": {"a3m": {"alignment": f">seq\n{sequence}", "format": "a3m"}}}
        }],
        "ligands": [{"id": "L1", "smiles": ligand_smiles, "predict_affinity": True}],
        "sampling_steps": 50, "diffusion_samples": 3
    }
    response = requests.post(PUBLIC_URL, headers=HEADERS, json=request_body)
    if response.status_code == 429:
        time.sleep(1)
        return predict_structure(protein_id, sequence, ligand_smiles, template_b64)
    return response.json()
# Prepare templates
protein_sequences = [
    {
        "id": "PolH",
        "sequence": "MATGQDRVVALVDMDCFFVQVEQRQNPHLRNKPCAVVQYKSWKGGGIIAVSYEARAFGVTRSMWADDAKKLCPDLLLAQVRESRGKANLTKYREASVEVMEIMSRFAVIERASIDEAYVDLTSAVQERLQKLQGQPISADLLPSTYIEGLPQGPTTAEETVQKEGMRKQGLFQWLDSLQIDNLTSPDLQLTVGAVIVEEMRAAIERETGFQCSAGISHNKVLAKLACGLNKPNRQTLVSHGSVPQLFSQMPIRKIRSLGGKLGASVIEILGIEYMGELTQFTESQLQSHFGEKNGSWLYAMCRGIEHDPVKPRQLPKTIGCSKNFPGKTALATREQVQWWLLQLAQELEERLTKDRNDNDRVATQLVVSIRVQGDKRLSSLRRCCALTRYDAHKMSHDAFTVIKNCNTSGIQTEWSPPLTMLFLCATKFSASAPSSSTDITSFLSSDPSSLPKVPVTSSEAKTQGSGPAVTATKKATTSLESFFQKAAERQKVKEASLSSLTAPTQAPMSNSPSKPSLPFQTSQSTGTEPFFKQKSLLLKQKQLNNSSVSSPQQNPWSNCKALPNSLPTEYPGCVPVCEGVSKLEESSKATPAEMDLAHNSQSMHASSASKSVLEVTQKATPNPSLLAAEDQVPCEKCGSLVPVWDMPEHMDYHFALELQKSFLQPHSSNPQVVSAVSHQGKRNPKSPLACTNKRPRPEGMQTLESFFKPLTH"
    },
    {
        "id": "NFE2L2",
        "sequence": "MMDLELPPPGLPSQQDMDLIDILWRQDIDLGVSREVFDFSQRRKEYELEKQKKLEKERQEQLQKEQEKAFFAQLQLDEETGEFLPIQPAQHIQSETSGSANYSQVAHIPKSDALYFDDCMQLLAQTFPFVDDNEVSSATFQSLVPDIPGHIESPVFIATNQAQSPETSVAQVAPVDLDGMQQDIEQVWEELLSIPELQCLNIENDKLVETTMVPSPEAKLTEVDNYHFYSSIPSMEKEVGNCSPHFLNAFEDSFSSILSTEDPNQLTVNSLNSDATVNTDFGDEFYSAFIAEPSISNSMPSPATLSHSLSELLNGPIDVSDLSLCKAFNQNHPESTAEFNDSDSGISLNTSPSVASPEHSVESSSYGDTLLGLSDSEVEELDSAPGSVKQNGPKTPVHSSGDMVQPLSPSQGQSTHVHDAQCENTPEKELPVSPGHRKTPFTKDKHSSRLEAHLTRDELRAKALHIPFPVEKIINLPVVDFNEMMSKEQFNEAQLALIRDIRRRGKNKVAAQNCRKRKLENIVELEQDLDHLKDEKEKLLKEKGENDKSLHLLKKQLSTLYLEVFSMLRDEDGKPYSPSEYSLQQTRDGNVFLVPKSKKPDVKKN"
    },
    {
        "id": "PolB",
        "sequence": "MSKRKAPQETLNGGITDMLTELANFEKNVSQAIHKYNAYRKAASVIAKYPHKIKSGAEAKKLPGVGTKIAEKIDEFLATGKLRKLEKIRQDDTSSSINFLTRVSGIGPSAARKFVDEGIKTLEDLRKNEDKLNHHQRIGLKYFGDFEKRIPREEMLQMQDIVLNEVKKVDSEYIATVCGSFRRGAESSGDMDVLLTHPSFTSESTKQPKLLHQVVEQLQKVHFITDTLSKGETKFMGVCQLPSKNDEKEYPHRRIDIRLIPKDQYYCGVLYFTGSDIFNKNMRAHALEKGFTINEYTIRPLGVTGVAGEPLPVDSEKDIFDYIQWKYREPKDRSE"
    },
    {
        "id": "APEX1",
        "sequence": "MPKRGKKGAVAEDGDELRTEPEAKKSKTAAKKNDKEAAGEGPALYEDPPDQKTSPSGKPATLKICSWNVDGLRAWIKKKGLDWVKEEAPDILCLQETKCSENKLPAELQELPGLSHQYWSAPSDKEGYSGVGLLSRQCPLKVSYGIGDEEHDQEGRVIVAEFDSFVLVTAYVPNAGRGLVRLEYRQRWDEAFRKFLKGLASRKPLVLCGDLNVAHEEIDLRNPKGNKKNAGFTPQERQGFGELLQAVPLADSFRHLYPNTPYAYTFWTYMMNARSKNVGWRLDYFLLSHSLLPALCDSKIRSKALGSDHCPITLYLAL"
    },
    {
        "id": "TDP1",
        "sequence": "MAKDAGLIEANGELKVFIDQNLSPGKGVVSLVAVHPSTVNPLGKQLLPKTFGQSNVNIAQQVVIGTPQRPAASNTLVVGSPHTPSTHFASQNQPSDSSPWSAGKRNRKGEKNGKGLRHFSMKVCEKVQRKGTTSYNEVADELVAEFSAADNHILPNESAYDQKNIRRRVYDALNVLMAMNIISKEKKEIKWIGLPTNSAQECQNLEVERQRRLERIKQKQSQLQELILQQIAFKNLVQRNRHAEQQASRPPPPNSVIHLPFIIVNTSKKTVIDCSISNDKFEYLFNFDNTFEIHDDIEVLKRMGMACGLESGSCSAEDLKMARSLVPKALEPYVTEMAQGTVGGVFITTAGSTSNGTRFSASDLTNGADGMLATSSNGSQYSGSRVETPVSYVGEDDEEDDDFNENDEDD"
    },
    {
        "id": "KDM4E",
        "sequence": "MKSVHSSPQNTSHTIMTFYPTMEEFADFNTYVAYMESQGAHQAGLAKVIPPKEWKARQMYDDIEDILIATPLQQVTSGQGGVFTQYHKKKKAMRVGQYRRLANSKKYQTPPHQNFADLEQRYWKSHPGNPPIYGADISGSLFEESTKQWNLGHLGTILDLLEQECGVVIEGVNTPYLYFGMWKTTFAWHTEDMDLYSINYLHFGEPKTWYVVPPEHGQHLERLARELFPDISRGCEAFLRHKVALISPTVLKENGIPFNCMTQEAGEFMVTFPYGYHAGFNHGFNCAEAINFATPRWIDYGKMASQCSCGESTVTFSMDPFVRIVQPESYELWKHRQDLAIVEHTEPRVAESQELSNWRDDIVLRRAALGLRLLPNLTAQCPTQPVSSGHCYNPKGCGTDAVPGSAFQSSAYHTQTQSLTLGMSARVLLPSTGSWGSGRGRGRGQGQGRGCSRGRGHGCCTRELGTEEPTVQPASKRRLLMGTRSRAQGHRPQLPLANDLMTNLSL"
    },
    {
        "id": "CYP2D6",
        "sequence": "MGLEALVPLAVIVAIFLLLVDLMHRRQRWAARYPPGPLPLPGLGNLLHVDFQNTPYCFDQLRRRFGDVFSLQLAWTPVVVLNGLAAVREALVTHGEDTADRPPVPITQILGFGPRSQGVFLARYGPAWREQRRFSVSTLRNLGLGKKSLEQWVTEEAACLCAAFANHSGRPFRPNGLLDKAVSNVIASLTCGRRFEYDDPRFLRLLDLAQEGLKEESGFLREVLNAVPVLLHIPALAGKVLRFQKAFLTQLDELLTEHRMTWDPAQPPRDLTEAFLAEMEKAKGNPESSFNDENLRIVVADLFSAGMVTTSTTLAWGLLLMILHPDVQRRVQQEIDDVIGQVRRPEMGDQAHMPYTTAVIHEVQRFGDIVPLGVTHMTSRDIEVQGFRIPKGTTLITNLSSVLKDEAVWEKPFRFHPEHFLDAQGHFVKPEAFLPFSAGRRACLGEPLARMELFLFFTSLLQHFSFSVPTGQPRPSHHGVFAFLVSPSPYELCAVPR"
    },
    {
        "id": "CYP3A4",
        "sequence": "MALIPDLAMETWLLLAVSLVLLYLYGTHSHGLFKKLGIPGPTPLPFLGNILSYHKGFCMFDMECHKKYGKVWGFYDGQQPVLAITDPDMIKTVLVKECYSVFTNRRPFGPVGFMKSAISIAEDEEWKRLRSLLSPTFTSGKLKEMVPIIAQYGDVLVRNLRREAETGKPVTLKDVFGAYSMDVITSTSFGVNIDSLNNPQDPFVENTKKLLRFDFLDPFFLSITVFPFLIPILEVLNICVFPREVTNFLRKSVKRMKESRLEDTQKHRVDFLQLMIDSQNSKETESHKALSDLELVAQSIIFIFAGYETTSSVLSFIMYELATHPDVQQKLQEEIDAVLPNKAPPTYDTVLQMEYLDMVVNETLRLFPIAMRLERVCKKDVEINGMFIPKGVVVMIPSYALHRDPKYWTEPEKFLPERFSKKNKDNIDPYIYTPFGSGPRNCIGMRFALMNMKLALIRVLQNFSFKPCKETQIPLKLSLGGLLQPEKPVVLKVESRDGTVSGA"
    },
    {
        "id": "ALDOA",
        "sequence": "MPYQYPALTPEQKKELSDIAHRIVAPGKGILAADESTGSIAKRLQSIGTENTEENRRFYRQLLLTADDRVNPCIGGVILFHETLYQKADDGRPFPQVIKSKGGVVGIKVDKGVVPLAGTNGETTTQGLDGLSERCAQYKKDGADFAKWRCVLKIGEHTPSALAIMENANVLARYASICQQNGIVPIVEPEILPDGDHDLKRCQYVTEKVLAAVYKALSDHHIYLEGTLLKPNMVTPGHACTQKFSHEEIAMATVTALRRTVPPAVTGITFLSGGQSEEEASINLNAINKCPLLKPWALTFSYGRALQASALKAWGGKKENLKAAQEEYVKRALANSLACQGKYTPSGQAGAAASESLFVSNHAY"
    },
    {
        "id": "KDM4A",
        "sequence": "MASESETLNPSARIMTFYPTMEEFRNFSRYIAYIESQGAHRAGLAKVVPPKEWKPRASYDDIDDLVIPAPIQQLVTGQSGLFTQYNIQKKAMTVREFRKIANSDKYCTPRYSEFEELERKYWKNLTFNPPIYGADVNGTLYEKHVDEWNIGRLRTILDLVEKESGITIEGVNTPYLYFGMWKTSFAWHTEDMDLYSINYLHFGEPKSWYSVPPEHGKRLERLAKGFFPGSAQSCEAFLRHKMTLISPLMLKKYGIPFDKVTQEAGEFMITFPYGYHAGFNHGFNCAESTNFATRRWIEYGKQAVLCSCRKDMVKISMDVFVRKFQPERYKLWKAGKDNTVIDHTLPTPEAAEFLKESELPPRAGNEEECPEEDMEGVEDGEEGDLKTSLAKHRIGTKRHRVCLEIPQEVSQSELFPKEDLSSEQYEMTECPAALAPVRPTHSSVRQVEDGLTFPDYSDSTEVKFEELKNVKLEEEDEEEEQAAAALDLSVNPASVGGRLVFSGSKKKSSSSLGSGSSRDSISSDSETSEPLSCRAQGQTGVLTVHSYAKGDGRVTVGEPCTRKKGSAARSFSERELAEVADEYMFSLEENKKSKGRRQPLSKLPRHHPLVLQECVSDDETSEQLTPEEEAEETEAWAKPLSQLWQNRPPNFEAEKEFNETMAQQAPHCAVCMIFQTYHQVEFGGFNQNCGNASDLAPQKQRTKPLIPEMCFTSTGCSTDINLSTPYLEEDGTSILVSCKKCSVRVHASCYGVPPAKASEDWMCSRCSANALEEDCCLCSLRGGALQRANDDRWVHVSCAVAILEARFVNIAERSPVDVSKIPLPRFKLKCIFCKKRRKRTAGCCVQCSHGRCPTAFHVSCAQAAGVMMQPDDWPFVVFITCFRHKIPNLERAKGALQSITAGQKVISKHKNGRFYQCEVVRLTTETFYEVNFDDGSFSDNLYPEDIVSQDCLQFGPPAEGEVVQVRWTDGQVYGAKFVASHPIQMYQVEFEDGSQLVVKRDDVYTLDEELPKRVKSRLSVASDMRFNEIFTEKEVKQEKKRQRVINSRYREDYIEPALYRAIME"
    }
]

template_pdb_map = {
    # AOP proteins
    "PolH": "3MR3",
    "NFE2L2": "2FLU",
    "PolB": "1BPX",
    "APEX1": "1BIX",
    "TDP1": "2AZE",
    "KDM4E": "2W2I",
    "CYP2D6": "2F9Q",
    "CYP3A4": "1TQN",
    "ALDOA": "1ALD",
    "KDM4A": "2GF7"
}
template_b64_map = {p_id: load_pdb_base64(download_pdb(pdb)) for p_id, pdb in template_pdb_map.items()}

output_dir = Path("boltz2_batch_outputs")
output_dir.mkdir(exist_ok=True)

all_features = []
for protein in protein_sequences:
    p_id = protein["id"]
    print(f"Processing: {p_id}")
    try:
        # Get raw result from API
        raw_result = predict_structure(p_id, protein["sequence"], ligand, template_b64_map.get(p_id))

        # --- NEW: SAVE RAW JSON TO DISK ---
        # This is what the visualizer is looking for
        json_path = output_dir / f"{p_id}.json"
        with open(json_path, "w") as f:
            json.dump(raw_result, f)

        # Extract features for the NN
        feats = extract_features_from_result(raw_result)
        if feats:
            feats["id"] = p_id
            all_features.append(feats)
            print(f"Success: pIC50={feats['boltz_pic50']:.2f} (Saved to {json_path})")

    except Exception as e:
        print(f"Failed {p_id}: {e}")

# --- 5. NN FILTERING & BINARY CLASSIFICATION ---
def process_features(df, smiles):
    mol = Chem.MolFromSmiles(smiles)
    lig_desc = {
        "mol_wt": Descriptors.MolWt(mol) if mol else 0,
        "logp": Descriptors.MolLogP(mol) if mol else 0,
        "h_donors": Descriptors.NumHDonors(mol) if mol else 0,
        "h_acceptors": Descriptors.NumHAcceptors(mol) if mol else 0,
        "tpsa": Descriptors.TPSA(mol) if mol else 0,
        "rotatable_bonds": Descriptors.NumRotatableBonds(mol) if mol else 0,
    }

    # Feature Mapping
    df["contact_bin_0_2.5"] = df["bin_0_2.5"]
    df["contact_bin_2.5_3.5"] = df["bin_2.5_3.5"]
    df["contact_bin_3.5_4.5"] = df["bin_3.5_4.5"]
    df["contact_bin_4.5_6"] = df["bin_4.5_6.0"]
    df["strong_contacts"] = df["bin_0_2.5"]
    df["weak_contacts"] = df["bin_2.5_3.5"]
    df["contact_density"] = df["num_contacts"] / (df["confidence"] + 1e-6)
    df["affinity_contact_ratio"] = df["boltz_pic50"] * df["confidence"]
    for k, v in lig_desc.items(): df[k] = v

    base_features = [
        "boltz_pic50", "boltz_prob", "confidence",
        "contact_bin_0_2.5", "contact_bin_2.5_3.5", "contact_bin_3.5_4.5", "contact_bin_4.5_6",
        "strong_contacts", "weak_contacts", "contact_density", "affinity_contact_ratio",
        "mol_wt", "logp", "h_donors", "h_acceptors", "tpsa", "rotatable_bonds"
    ]

    X_raw = df[base_features].values
    log_indices = [3, 4, 5, 6, 7, 8, 9, 10, 11, 15]
    X_raw[:, log_indices] = np.log1p(np.maximum(X_raw[:, log_indices], 0))

    poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
    X_inter = poly.fit_transform(X_raw)
    return StandardScaler().fit_transform(X_inter)

# Clean the string (remove any accidental whitespace/newlines from pasting)
MODEL_B64 = "".join(MODEL_B64.split())

# Apply padding fix
missing_padding = len(MODEL_B64) % 4
if missing_padding:
    MODEL_B64 += "=" * (4 - missing_padding)

try:
    print("Decoding model weights...")
    model_bytes = base64.b64decode(MODEL_B64)
    buffer = io.BytesIO(model_bytes)

    # Initialize Model
    model = BinderNet(input_dim=153)

    # Load state dict
    state_dict = torch.load(buffer, map_location=torch.device('cpu'))

    # Handle both direct state_dicts and nested 'model_state_dict' (common in checkpoints)
    if 'model_state_dict' in state_dict:
        model.load_state_dict(state_dict['model_state_dict'])
    else:
        model.load_state_dict(state_dict)

    model.eval()
    print("Model loaded successfully.")

    if all_features:
        inference_df = pd.DataFrame(all_features)
        X_scaled = process_features(inference_df, ligand)

        with torch.no_grad():
            X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
            probs = model(X_tensor).numpy().flatten()

            inference_df["nn_score"] = probs
            inference_df["is_hit"] = (probs >= 0.5).astype(int)

        print(f"\nFound {inference_df['is_hit'].sum()} Predicted Binders.")
        display(inference_df[['id', 'boltz_pic50', 'clash_count', 'nn_score', 'is_hit']].sort_values("nn_score", ascending=False))
    else:
        print("No features extracted from Boltz-2 results. Cannot run NN inference.")

except Exception as e:
    print(f"Model Loading Error: {e}")
    print("\nTroubleshooting Tip: If you see 'Central Directory' errors, the string is still missing characters at the end.")

Processing: PolH
Success: pIC50=6.07 (Saved to boltz2_batch_outputs/PolH.json)
Processing: NFE2L2
Success: pIC50=6.40 (Saved to boltz2_batch_outputs/NFE2L2.json)
Processing: PolB
Success: pIC50=5.91 (Saved to boltz2_batch_outputs/PolB.json)
Processing: APEX1
Success: pIC50=5.61 (Saved to boltz2_batch_outputs/APEX1.json)
Processing: TDP1
Success: pIC50=6.22 (Saved to boltz2_batch_outputs/TDP1.json)
Processing: KDM4E
Success: pIC50=6.37 (Saved to boltz2_batch_outputs/KDM4E.json)
Processing: CYP2D6
Success: pIC50=6.59 (Saved to boltz2_batch_outputs/CYP2D6.json)
Processing: CYP3A4
Success: pIC50=6.44 (Saved to boltz2_batch_outputs/CYP3A4.json)
Processing: ALDOA
Success: pIC50=6.50 (Saved to boltz2_batch_outputs/ALDOA.json)
Processing: KDM4A
Success: pIC50=6.68 (Saved to boltz2_batch_outputs/KDM4A.json)
Decoding model weights...
Model loaded successfully.

Found 3 Predicted Binders.


,id,boltz_pic50,clash_count,nn_score,is_hit
2,PolB,5.914219,0,0.706802,1
7,CYP3A4,6.436375,0,0.590148,1
9,KDM4A,6.681469,0,0.556015,1
3,APEX1,5.605188,0,0.429382,0
5,KDM4E,6.372438,0,0.397788,0
8,ALDOA,6.500313,0,0.227614,0
6,CYP2D6,6.585563,0,0.124988,0
4,TDP1,6.223250,0,0.030406,0
1,NFE2L2,6.404406,0,0.021613,0
0,PolH,6.074063,1,0.003213,0
